# Landing to Bronze — Projeto V-Energy

Ingestão de arquivos CSV, JSON e XLSX da landing zone (UC Volume) para a camada Bronze do catálogo `v-energy`, seguindo a Arquitetura Medalhão.

In [0]:
catalog = "v-energy"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"`{catalog}`.{bronze_schema_name}"
silver_schema = f"`{catalog}`.{silver_schema_name}"
gold_schema = f"`{catalog}`.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}\n")

catalog: v-energy
bronze_schema: `v-energy`.bronze
silver_schema: `v-energy`.silver
gold_schema: `v-energy`.gold
landing_path: /Volumes/v-energy/bronze/landing



## 2. Criação do Catalog, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{bronze_schema_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{silver_schema_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{gold_schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalog}`.{bronze_schema_name}.landing")

print("✅ Catalog, schemas e volume prontos.\n")

✅ Catalog, schemas e volume prontos.



## 3. Validação da Landing Zone

Sete arquivos têm nome fixo. As leituras entram por `leituras_*.csv`. Se faltar um nome fixo ou um ano obrigatório de leitura (`2022` a `2026`), o notebook para. Outro CSV é ignorado.

In [0]:
# Fontes de nome fixo: (nome, linhas esperadas, tipo).
# Leituras não entram aqui: o ano novo chega como outro arquivo leituras_*.csv.
arquivos_fixos = [
    ("unidades_consumidoras.csv", 46461, "csv"),
    ("medidores.csv", 51812, "csv"),
    ("ocorrencias_rede.csv", 180000, "csv"),
    ("faturas.csv", 2184623, "csv"),
    ("geracao_distribuida.csv", 3510, "csv"),
    ("atendimentos.json", 318871, "json"),
    ("inspecoes.xlsx", 1389, "xlsx"),
]

# O arquivo destes anos têm de existir, com estas contagens.
# leituras_2027.csv, por exemplo, entra sem contagem contratada.
leituras_obrigatorias = {
    "leituras_2022.csv": 271744,
    "leituras_2023.csv": 544546,
    "leituras_2024.csv": 545652,
    "leituras_2025.csv": 546908,
    "leituras_2026.csv": 273618,
}

nomes_fixos = {nome for nome, _, _ in arquivos_fixos}

try:
    arquivos_no_volume = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    arquivos_no_volume = set()
    print(f"[ERRO] Não foi possível listar '{landing_path}': {e}")

leituras_no_volume = sorted(
    nome for nome in arquivos_no_volume
    if nome.startswith("leituras_") and nome.endswith(".csv")
)
ausentes = (nomes_fixos - arquivos_no_volume) | (set(leituras_obrigatorias) - set(leituras_no_volume))
if ausentes:
    print("❌ [ERRO] Arquivos esperados não encontrados na landing zone:")
    for nome in sorted(ausentes):
        print(f"   - {nome}")
    raise SystemExit("Notebook interrompido: há arquivos ausentes na landing zone.")

leituras_para_ingerir = []
for nome in leituras_no_volume:
    if nome in leituras_obrigatorias:
        leituras_para_ingerir.append((nome, leituras_obrigatorias[nome], "csv"))
    else:
        leituras_para_ingerir.append((nome, None, "csv"))
        print(f"⚠️ [AVISO] {nome} entra pelo padrão leituras_*.csv, sem contagem contratada.")

extras = arquivos_no_volume - nomes_fixos - set(leituras_no_volume)
if extras:
    print("⚠️ [AVISO] Arquivos ignorados (fora da lista e fora de leituras_*.csv):")
    for nome in sorted(extras):
        print(f"   - {nome}")

arquivos_esperados = arquivos_fixos + leituras_para_ingerir

print(f"\n✅ [OK] {len(arquivos_fixos)} arquivo(s) de nome fixo e {len(leituras_para_ingerir)} leitura(s).")
for nome, esperado, tipo in arquivos_esperados:
    if esperado is None:
        print(f"   - {nome} ({tipo}) — sem contagem contratada")
    else:
        print(f"   - {nome} ({tipo}) — esperado: {esperado:,} linhas")


✅ [OK] 7 arquivo(s) de nome fixo e 5 leitura(s).
   - unidades_consumidoras.csv (csv) — esperado: 46,461 linhas
   - medidores.csv (csv) — esperado: 51,812 linhas
   - ocorrencias_rede.csv (csv) — esperado: 180,000 linhas
   - faturas.csv (csv) — esperado: 2,184,623 linhas
   - geracao_distribuida.csv (csv) — esperado: 3,510 linhas
   - atendimentos.json (json) — esperado: 318,871 linhas
   - inspecoes.xlsx (xlsx) — esperado: 1,389 linhas
   - leituras_2022.csv (csv) — esperado: 271,744 linhas
   - leituras_2023.csv (csv) — esperado: 544,546 linhas
   - leituras_2024.csv (csv) — esperado: 545,652 linhas
   - leituras_2025.csv (csv) — esperado: 546,908 linhas
   - leituras_2026.csv (csv) — esperado: 273,618 linhas


## 4. Ingestão de Dados na Bronze

### Architecture Decision Record (ADR): Tratamento de I/O na Ingestão (multiLine e escape)

**Contexto e Decisão:**
Para garantir o princípio arquitetural da camada Bronze de "evitar perda de dados na origem", as opções `.option("multiLine", "true")` e `.option("escape", '"')` estão ativas para **todos** os arquivos CSV. O objetivo é blindar o parser do Spark contra o *Column Shift* (deslocamento de colunas) causado por quebras de linha irregulares e aspas aninhadas em campos de texto livre.

**Justificativa Técnica:**
1. **Sem proteção (Inflação e Corrupção):** O Spark quebra os registros a cada `\n` acidental no meio de um texto, inflando a base artificialmente com "linhas fantasmas" e deslocando as colunas à direita.
2. **Apenas `multiLine` (Perda Silenciosa):** O Spark se perde em aspas duplas aninhadas (`""`), assumindo finais de texto incorretos e "engolindo" registros ao aglutinar múltiplas linhas numa única.
3. **Aplicação em Todas as Tabelas:** Manter a blindagem para todos os arquivos garante a integridade total da camada Bronze e previne a quebra do pipeline por viés de amostragem.

> **Nota:** A comprovação está no Teste 1 da seção 5. Nos 10 CSVs, as quatro combinações deram a mesma contagem, e essa contagem fecha com a Bronze.

In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pyspark.sql.functions as F
import re
import os
from pyspark.sql.types import ArrayType, StringType, StructField, StructType

def detect_delimiter(file_path):
    """Escolhe o separador do arquivo inteiro pelo cabeçalho.

    Lê só a primeira linha. Se ';' aparece mais que ',', o separador é ';'.
    Nos arquivos desta entrega o outro caractere está em zero: faturas.csv
    usa ';', os demais CSVs usam ','. Vírgula dentro de valor (R$ 225,53
    ou "5,03") não entra na conta, porque essa linha não é lida aqui.
    """
    try:
        first_line = dbutils.fs.head(file_path, 4096).split("\n")[0]
        if first_line.count(";") > first_line.count(","):
            return ";"
        return ","
    except Exception:
        return ","

def sanitize_column_name(name):
    """Sanitiza nomes de colunas para serem válidos no Delta (remove ' ,;{}()\n\t=')."""
    sanitized = re.sub(r'[ ,;{}()\n\t=]', '_', name.strip())
    return sanitized if sanitized else "_unnamed"

def sanitize_columns(df):
    """Sanitiza nomes de colunas de um DataFrame Spark (toDF evita loop de withColumnRenamed)."""
    new_names = [sanitize_column_name(c) for c in df.columns]
    if list(df.columns) != new_names:
        df = df.toDF(*new_names)
    return df

def derive_table_name(filename):
    """Deriva o nome da tabela Bronze a partir do nome do arquivo."""
    nome_base = os.path.splitext(filename)[0]
    nome_tabela = nome_base.replace(" ", "_").replace("-", "_").lower()
    if not nome_tabela.startswith("tb_"):
        nome_tabela = f"tb_{nome_tabela}"
    return nome_tabela

def schema_texto(colunas):
    return StructType([StructField(c, StringType(), True) for c in colunas])

# Contrato de cabeçalho. Se a origem renomear ou reordenar, a carga falha
# em vez de gravar coluna deslocada. Tudo fica string.
CONTRATOS_CSV = {
    "unidades_consumidoras.csv": [
        "id_uc", "cpf_cnpj", "nome_titular", "logradouro", "numero", "bairro",
        "municipio", "uf", "cep", "classe", "subgrupo_tarifario", "alimentador",
        "data_ligacao", "status", "latitude", "longitude",
    ],
    "medidores.csv": [
        "id_medidor", "id_uc", "modelo", "tipo", "data_instalacao",
        "data_retirada", "data_ultima_manutencao", "status",
    ],
    "ocorrencias_rede.csv": [
        "id_ocorrencia", "alimentador", "id_uc", "tipo", "causa",
        "data_abertura", "data_encerramento", "duracao_min",
    ],
    "faturas.csv": [
        "id_fatura", "id_uc", "mes_referencia", "data_emissao", "valor_energia",
        "tributos", "bandeira_tarifaria", "valor_total", "data_vencimento",
        "data_pagamento", "status",
    ],
    "geracao_distribuida.csv": [
        "id_uc", "potencia_kwp", "data_conexao", "modalidade",
        "energia_injetada_kwh_media_mensal",
    ],
}
COLUNAS_LEITURA = [
    "id_leitura", "id_uc", "id_medidor", "data_leitura", "mes_referencia",
    "leitura_anterior", "leitura_atual", "consumo_kwh", "tipo_leitura",
]

def ler_csv(caminho, colunas):
    """Lê CSV como texto, confere o cabeçalho e derruba linha malformada."""
    delimiter = detect_delimiter(caminho)
    df = (
        spark.read.format("csv")
        .schema(schema_texto(colunas))
        .option("header", "true")
        .option("sep", delimiter)
        .option("encoding", "UTF-8")
        .option("enforceSchema", "false")
        .option("mode", "FAILFAST")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("ignoreLeadingWhiteSpace", "false")
        .option("ignoreTrailingWhiteSpace", "false")
        .load(caminho)
        .withColumn("source_file", F.col("_metadata.file_name"))
    )
    return df, delimiter

def conferir_contagem(rotulo, obtido, esperado):
    """Menos linhas do que o esperado interrompe a carga. Mais linhas avisa e segue."""
    if obtido is None:
        raise SystemExit(f"[ERRO] {rotulo}: nenhuma linha lida. A tabela não foi substituída.")
    if esperado is None:
        print(f"      Aviso: {rotulo} sem contagem contratada. Lidas {obtido:,} linhas.")
        return
    if obtido < esperado:
        raise SystemExit(
            f"[ERRO] {rotulo}: esperado pelo menos {esperado:,} linhas, obtido {obtido:,}. A tabela não foi substituída."
        )
    if obtido > esperado:
        print(f"      Aviso: {rotulo}: esperado {esperado:,} linhas, obtido {obtido:,}. A carga segue.")

def validar_chaves_json(df, nome_arquivo):
    """Chave obrigatória ausente interrompe a carga. nota_satisfacao pode ser nula. Ordem não importa."""
    obrigatorias = [
        "protocolo", "id_uc", "canal", "assunto", "subassunto",
        "data_abertura", "data_fechamento", "nota_satisfacao", "historico",
    ]
    ausentes = [c for c in obrigatorias if c not in df.columns]
    if ausentes:
        raise SystemExit(
            f"[ERRO] {nome_arquivo}: chaves ausentes: {ausentes}. A tabela não foi substituída."
        )
    tipo_hist = df.schema["historico"].dataType
    if not isinstance(tipo_hist, ArrayType) or not isinstance(tipo_hist.elementType, StructType):
        raise SystemExit(
            f"[ERRO] {nome_arquivo}: historico não é uma lista de objetos. A tabela não foi substituída."
        )
    sub_obrigatorias = ["sequencia", "data", "acao", "canal"]
    campos_hist = [campo.name for campo in tipo_hist.elementType.fields]
    sub_ausentes = [c for c in sub_obrigatorias if c not in campos_hist]
    if sub_ausentes:
        raise SystemExit(
            f"[ERRO] {nome_arquivo}: subchaves ausentes em historico: {sub_ausentes}. A tabela não foi substituída."
        )
    sem_valor = [
        "protocolo", "id_uc", "canal", "assunto", "subassunto",
        "data_abertura", "data_fechamento",
    ]
    cond = F.col("historico").isNull() | (F.size("historico") == 0)
    for coluna in sem_valor:
        cond = cond | F.col(coluna).isNull()

    def item_incompleto(item):
        ruim = item["sequencia"].isNull()
        for nome in ("data", "acao", "canal"):
            ruim = ruim | item[nome].isNull()
        return ruim

    cond = cond | F.exists("historico", item_incompleto)
    if df.filter(cond).limit(1).count():
        raise SystemExit(
            f"[ERRO] {nome_arquivo}: há chamado sem chave obrigatória, com historico vazio ou com subchave vazia. "
            "nota_satisfacao pode ser nula. A tabela não foi substituída."
        )

def gravar_bronze(df, tabela_destino):
    (
        df.withColumn("ingestion_datetime", F.current_timestamp())
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

leituras = [item for item in arquivos_esperados if item[0].startswith("leituras_")]
outros = [item for item in arquivos_esperados if not item[0].startswith("leituras_")]

print(f"Iniciando ingestão de {len(outros)} fonte(s) de nome fixo e {len(leituras)} leitura(s)...\n")

for nome_arquivo, esperado, tipo in outros:
    caminho_arquivo = f"{landing_path}/{nome_arquivo}"
    nome_tabela = derive_table_name(nome_arquivo)
    tabela_destino = f"{bronze_schema}.{nome_tabela}"

    if tipo == "csv":
        colunas = CONTRATOS_CSV[nome_arquivo]
        df_raw, delimiter = ler_csv(caminho_arquivo, colunas)
        print(f"   Tipo: CSV | Separador: '{delimiter}' | {nome_arquivo}")
    elif tipo == "json":
        print(f"   Tipo: JSON | {nome_arquivo}")
        df_raw = (
            spark.read.option("multiLine", "true")
            .option("mode", "PERMISSIVE")
            .json(caminho_arquivo)
        )
        if "_corrupt_record" in df_raw.columns:
            raise SystemExit(f"[ERRO] {nome_arquivo}: há registro JSON que o leitor não conseguiu ler.")
        df_raw = df_raw.withColumn("source_file", F.lit(nome_arquivo))
        validar_chaves_json(df_raw, nome_arquivo)
    elif tipo == "xlsx":
        import pandas as pd
        print(f"   Tipo: XLSX (abas com cabeçalho de inspeção) | {nome_arquivo}")

        colunas_inspecao = [
            "id_inspecao", "id_uc", "data", "inspetor",
            "resultado", "tipo_irregularidade", "kwh_recuperado",
        ]
        abas_obrigatorias = {"ago-set": 511, "out-nov": 495, "dez": 383}

        xl = pd.ExcelFile(caminho_arquivo, engine="openpyxl")
        pdfs = []
        abas_lidas = {}
        for nome_aba in xl.sheet_names:
            pdf_aba = pd.read_excel(xl, sheet_name=nome_aba)
            colunas = [str(c).strip() for c in pdf_aba.columns]
            if colunas != colunas_inspecao:
                print(f"      Aba '{nome_aba}' ignorada: cabeçalho diferente de inspeção")
                continue
            pdf_aba.columns = colunas
            pdf_aba["sheet"] = nome_aba
            pdfs.append(pdf_aba)
            abas_lidas[nome_aba] = len(pdf_aba)
            print(f"      Aba '{nome_aba}': {len(pdf_aba)} linhas")

        faltando = [nome for nome in abas_obrigatorias if nome not in abas_lidas]
        if faltando:
            raise SystemExit(f"[ERRO] Abas obrigatórias ausentes em {nome_arquivo}: {faltando}")
        for nome_aba, esperado_aba in abas_obrigatorias.items():
            conferir_contagem(f"Aba '{nome_aba}'", abas_lidas[nome_aba], esperado_aba)
        if not pdfs:
            raise SystemExit(f"[ERRO] Nenhuma aba de inspeção em {nome_arquivo}.")

        pdf_combined = pd.concat(pdfs, ignore_index=True)
        df_raw = spark.createDataFrame(pdf_combined)
        df_raw = df_raw.select([F.col(c).cast("string").alias(c) for c in df_raw.columns])
        df_raw = df_raw.withColumn("source_file", F.lit(nome_arquivo))
        print(f"      Total XLSX: {len(pdf_combined)} linhas")
    else:
        print(f"   ⏭️ [SKIP] Tipo {tipo} não suportado: {nome_arquivo}\n")
        continue

    df_raw = sanitize_columns(df_raw)

    # Menos linhas do que o esperado interrompe antes do overwrite.
    # Mais linhas avisa e a gravação segue.
    obtido_df = df_raw.count()
    if tipo != "xlsx":
        conferir_contagem(nome_arquivo, obtido_df, esperado)

    gravar_bronze(df_raw, tabela_destino)

    if tipo == "xlsx":
        contagem_abas = {
            row["sheet"]: row["qtd"]
            for row in spark.table(tabela_destino).groupBy("sheet").count().withColumnRenamed("count", "qtd").collect()
        }
        for nome_aba, lido in abas_lidas.items():
            obtido_aba = contagem_abas.get(nome_aba)
            if obtido_aba != lido:
                raise SystemExit(
                    f"[ERRO] {nome_arquivo}, aba '{nome_aba}': lidas {lido} linhas, tabela com {obtido_aba}."
                )
        extras = sorted(set(contagem_abas) - set(abas_obrigatorias))
        if extras:
            detalhe = ", ".join(f"{nome}={contagem_abas[nome]}" for nome in extras)
            print(f"      Aviso: abas adicionais com cabeçalho de inspeção: {detalhe}")
        obtido = sum(contagem_abas.values())
    else:
        obtido = spark.table(tabela_destino).count()
        if obtido != obtido_df:
            raise SystemExit(f"[ERRO] {nome_arquivo}: DataFrame {obtido_df:,} linhas, tabela {obtido:,}.")

    print(f"✅ [SUCESSO] {nome_arquivo} -> {tabela_destino} ({obtido:,} linhas)\n")

if leituras:
    caminhos = [f"{landing_path}/{nome}" for nome, _, _ in leituras]
    for caminho in caminhos:
        separador = detect_delimiter(caminho)
        if separador != ",":
            raise SystemExit(f"[ERRO] {os.path.basename(caminho)}: separador '{separador}', esperado ','.")
    df_leituras = (
        spark.read.format("csv")
        .schema(schema_texto(COLUNAS_LEITURA))
        .option("header", "true")
        .option("sep", ",")
        .option("encoding", "UTF-8")
        .option("enforceSchema", "false")
        .option("mode", "FAILFAST")
        .option("multiLine", "true")
        .option("escape", '"')
        .option("ignoreLeadingWhiteSpace", "false")
        .option("ignoreTrailingWhiteSpace", "false")
        .load(caminhos)
        .withColumn("source_file", F.col("_metadata.file_name"))
    )
    contagem = {
        row["source_file"]: row["qtd"]
        for row in df_leituras.groupBy("source_file").count().withColumnRenamed("count", "qtd").collect()
    }
    print("   Leituras por arquivo:", contagem)
    for nome, esperado, _ in leituras:
        conferir_contagem(nome, contagem.get(nome), esperado)
    tabela_leituras = f"{bronze_schema}.tb_leituras"
    gravar_bronze(df_leituras, tabela_leituras)
    obtido = spark.table(tabela_leituras).count()
    esperado_total = sum(qtd for qtd in contagem.values())
    if obtido != esperado_total:
        raise SystemExit(f"[ERRO] tb_leituras: DataFrame {esperado_total:,} linhas, tabela {obtido:,}.")
    print(f"✅ [SUCESSO] leituras_*.csv -> {tabela_leituras} ({obtido:,} linhas)\n")
    for ano in range(2022, 2027):
        spark.sql(f"DROP TABLE IF EXISTS {bronze_schema}.tb_leituras_{ano}")

print("✅ Ingestão concluída — fontes de nome fixo e leituras_*.csv processadas.")

Iniciando ingestão de 12 arquivo(s) para a Camada Bronze...

[Truncated to first 4096 bytes]
   Tipo: CSV | Separador: ',' | unidades_consumidoras.csv
✅ [SUCESSO] unidades_consumidoras.csv -> `v-energy`.bronze.tb_unidades_consumidoras (46,461 linhas)

[Truncated to first 4096 bytes]
   Tipo: CSV | Separador: ',' | medidores.csv
✅ [SUCESSO] medidores.csv -> `v-energy`.bronze.tb_medidores (51,812 linhas)

[Truncated to first 4096 bytes]
   Tipo: CSV | Separador: ',' | ocorrencias_rede.csv
✅ [SUCESSO] ocorrencias_rede.csv -> `v-energy`.bronze.tb_ocorrencias_rede (180,000 linhas)

[Truncated to first 4096 bytes]
   Tipo: CSV | Separador: ';' | faturas.csv
✅ [SUCESSO] faturas.csv -> `v-energy`.bronze.tb_faturas (2,184,623 linhas)

[Truncated to first 4096 bytes]
   Tipo: CSV | Separador: ',' | geracao_distribuida.csv
✅ [SUCESSO] geracao_distribuida.csv -> `v-energy`.bronze.tb_geracao_distribuida (3,510 linhas)

   Tipo: JSON | atendimentos.json
✅ [SUCESSO] atendimentos.json -> `v-energy`.br

### Contagem esperada

A validação está na seção 3. Sete arquivos têm nome e contagem de referência desta entrega. As leituras seguem `leituras_*.csv` e são gravadas juntas em `tb_leituras`. Antes do `overwrite`, menos linhas do que a referência interrompe a carga e a tabela atual permanece. Mais linhas imprime um aviso e a gravação segue. Um arquivo novo do padrão `leituras_*.csv`, sem contagem no código, entra com aviso. No Excel, `ago-set` (511), `out-nov` (495) e `dez` (383) seguem a mesma regra. Uma aba nova com o mesmo cabeçalho entra e só gera aviso.

## 5. Validação dos Tratamentos de Ingestão

### Que tipos de problemas podem ocorrer na ingestão de CSV?

| Problema | O que acontece | Como detectar |
| --- | --- | --- |
| **Inflação de linhas** | Quebras de linha (`\n`) acidentais dentro de campos de texto criam “linhas fantasmas” | Contagem de linhas sem `multiLine` é maior que com `multiLine` |
| **Perda silenciosa de linhas** | Aspas duplas aninhadas (`""`) fazem o parser aglutinar múltiplos registros em uma só linha | Contagem sem `escape` é menor que com `escape` |
| **Column Shift** | Parsing incorreto desloca valores para colunas erradas | Nulos em colunas que não deveriam ser nulas; tipos de dados inconsistentes |
| **Nulos anômalos** | Parsing incorreto deixa campos vazios onde deveria haver valores | Alta % de nulos em colunas que não deveriam ter |

### O que vamos testar (os 10 CSVs da lista fixa)

1. **Comparação de contagem de linhas** — ler cada arquivo com 4 combinações de opções (`sem proteção`, `apenas multiLine`, `apenas escape`, `multiLine + escape`) e comparar as contagens
2. **Contagem de linhas raw vs Bronze** — comparar o número de linhas do arquivo texto original com a tabela Bronze
3. **Validação de schema e nulos** — verificar colunas, tipos, e padrões de nulos anômalos

In [0]:
# Teste 1: Comparação de contagem de linhas — TODOS os arquivos CSV
# Objetivo: Verificar se as opções multiLine e escape alteraram a contagem de linhas

def _detect_delimiter(file_path):
    """Escolhe o separador do arquivo inteiro pelo cabeçalho.

    Lê só a primeira linha. Se ';' aparece mais que ',', o separador é ';'.
    Nos arquivos desta entrega o outro caractere está em zero: faturas.csv
    usa ';', os demais CSVs usam ','. Vírgula dentro de valor (R$ 225,53
    ou "5,03") não entra na conta, porque essa linha não é lida aqui.
    """
    try:
        first_line = dbutils.fs.head(file_path, 4096).split("\n")[0]
        return ";" if first_line.count(";") > first_line.count(",") else ","
    except Exception:
        return ","

# Só os CSVs da lista fixa. Arquivo extra no volume não entra no teste.
arquivos_teste = {
    nome: _detect_delimiter(f"{landing_path}/{nome}")
    for nome, _, tipo in arquivos_esperados
    if tipo == "csv"
}

print("=" * 130)
print(f"TESTE 1: Comparação de contagem de linhas — {len(arquivos_teste)} arquivo(s)")
print("=" * 130)

resultados = []

for nome_arquivo, delimiter in sorted(arquivos_teste.items()):
    caminho = f"{landing_path}/{nome_arquivo}"
    print(f"\n📄 {nome_arquivo} (sep='{delimiter}')")
    
    # Contagem raw (texto puro)
    raw_lines = spark.read.text(caminho).count()
    raw_data = raw_lines - 1
    
    # 4 combinações de opções
    combinacoes = [
        ("Sem proteção",       {}),
        ("Apenas multiLine",   {"multiLine": "true"}),
        ("Apenas escape",      {"escape": '"'}),
        ("multiLine + escape", {"multiLine": "true", "escape": '"'}),
    ]
    
    counts = {}
    for nome_comb, opts in combinacoes:
        reader = spark.read.format("csv").option("header", "true").option("sep", delimiter)
        for k, v in opts.items():
            reader = reader.option(k, v)
        counts[nome_comb] = reader.load(caminho).count()
    
    if nome_arquivo.startswith("leituras_"):
        bronze_count = spark.table(f"{bronze_schema}.tb_leituras").filter(f"source_file = '{nome_arquivo}'").count()
    else:
        nome_tb = nome_arquivo.replace(".csv", "").replace(" ", "_").replace("-", "_").lower()
        if not nome_tb.startswith("tb_"):
            nome_tb = f"tb_{nome_tb}"
        bronze_count = spark.table(f"{bronze_schema}.{nome_tb}").count()
    
    todas_iguais = len(set(counts.values())) == 1
    print(f"   Raw={raw_data:,} | SemProt={counts['Sem proteção']:,} | mL={counts['Apenas multiLine']:,} | esc={counts['Apenas escape']:,} | mL+esc={counts['multiLine + escape']:,} | Bronze={bronze_count:,}")
    
    if not todas_iguais:
        print(f"   ⚠️ Diferença entre combinações!")
    
    # Status raw vs bronze
    if raw_data == bronze_count:
        print(f"   ✅ Raw == Bronze")
    elif raw_data * 2 == bronze_count:
        print(f"   ⚠️ Bronze = 2x Raw (duplicação por re-execução append)")
    else:
        print(f"   ⚠️ Raw={raw_data:,} vs Bronze={bronze_count:,} — verificar")
    
    resultados.append({
        "arquivo": nome_arquivo, "sep": delimiter, "raw": raw_data, "bronze": bronze_count,
        "sem_prot": counts["Sem proteção"], "ml": counts["Apenas multiLine"],
        "esc": counts["Apenas escape"], "ml_esc": counts["multiLine + escape"],
        "igual": todas_iguais,
    })

# Tabela resumo
print(f"\n{'='*130}")
print(f"{'Arquivo':<30} {'Sep':>3} {'Raw':>12} {'SemProt':>12} {'multiLine':>12} {'escape':>12} {'mL+esc':>12} {'Bronze':>12} {'4comb':>6} {'Raw=Brz':>8}")
print("-" * 130)
for r in resultados:
    match = "✅" if r["raw"] == r["bronze"] else ("2x" if r["raw"]*2 == r["bronze"] else "⚠️")
    igual = "✅" if r["igual"] else "⚠️"
    print(f"{r['arquivo']:<30} {r['sep']:>3} {r['raw']:>12,} {r['sem_prot']:>12,} {r['ml']:>12,} {r['esc']:>12,} {r['ml_esc']:>12,} {r['bronze']:>12,} {igual:>6} {match:>8}")
print("=" * 130)

[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
[Truncated to first 4096 bytes]
TESTE 1: Comparação de contagem de linhas — 10 arquivo(s)

📄 faturas.csv (sep=';')
   Raw=2,184,623 | SemProt=2,184,623 | mL=2,184,623 | esc=2,184,623 | mL+esc=2,184,623 | Bronze=2,184,623
   ✅ Raw == Bronze

📄 geracao_distribuida.csv (sep=',')
   Raw=3,510 | SemProt=3,510 | mL=3,510 | esc=3,510 | mL+esc=3,510 | Bronze=3,510
   ✅ Raw == Bronze

📄 leituras_2022.csv (sep=',')
   Raw=271,744 | SemProt=271,744 | mL=271,744 | esc=271,744 | mL+esc=271,744 | Bronze=271,744
   ✅ Raw == Bronze

📄 leituras_2023.csv (sep=',')
   Raw=544,546 | SemProt=544,546 | mL=544,546 | esc=544,546 | mL+esc=544,546 | Bronze=544,546
   ✅ Raw == Bronze

📄 leituras_2024.csv (sep=',')
   Raw=545,652 

In [0]:
# Teste 2: Validação de schema e nulos — TODAS as tabelas Bronze
# Objetivo: Verificar colunas, tipos e padrões de nulos anômalos

from pyspark.sql.functions import col, count, when, isnull

# Listar todas as tabelas Bronze dinamicamente
bronze_tables = spark.sql(f"SHOW TABLES IN {bronze_schema}").select("tableName").collect()
tabelas = sorted([row.tableName for row in bronze_tables])

print("=" * 80)
print(f"TESTE 2: Schema e nulos — {len(tabelas)} tabela(s) Bronze")
print("=" * 80)

problemas_nulos = []

for nome_tb in tabelas:
    tabela_full = f"{bronze_schema}.{nome_tb}"
    df = spark.table(tabela_full)
    total = df.count()
    
    print(f"\n📊 {nome_tb} — {total:,} linhas, {len(df.columns)} colunas")
    
    # Schema (compacto)
    cols_info = ", ".join([f"{c}:{df.schema[c].dataType.simpleString()}" for c in df.columns])
    print(f"   {cols_info}")
    
    # Análise de nulos
    null_counts = df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).collect()[0]
    nulos_info = []
    for c in df.columns:
        n = null_counts[c]
        pct = (n / total * 100) if total > 0 else 0
        if pct > 0:
            nulos_info.append(f"{c}={n:,}({pct:.1f}%)")
        if pct > 10:
            problemas_nulos.append((nome_tb, c, n, pct))
    
    if nulos_info:
        print(f"   Nulos: {', '.join(nulos_info)}")
    else:
        print(f"   Nulos: ✅ 0 em todas as colunas")

# Resumo de nulos anômalos
print(f"\n{'='*80}")
if problemas_nulos:
    print("⚠️ Nulos anômalos (>10%):")
    for tb, c, n, p in problemas_nulos:
        print(f"   {tb}.{c}: {n:,} ({p:.1f}%)")
else:
    print("✅ Nenhuma coluna com >10% de nulos em nenhuma tabela")
print("=" * 80)

TESTE 2: Schema e nulos — 12 tabela(s) Bronze

📊 tb_atendimentos — 318,871 linhas, 10 colunas
   assunto:string, canal:string, data_abertura:string, data_fechamento:string, historico:array<struct<acao:string,canal:string,data:string,sequencia:bigint>>, id_uc:string, nota_satisfacao:double, protocolo:string, subassunto:string, ingestion_datetime:timestamp
   Nulos: nota_satisfacao=9,566(3.0%)

📊 tb_faturas — 2,184,623 linhas, 12 colunas
   id_fatura:string, id_uc:string, mes_referencia:string, data_emissao:string, valor_energia:string, tributos:string, bandeira_tarifaria:string, valor_total:string, data_vencimento:string, data_pagamento:string, status:string, ingestion_datetime:timestamp
   Nulos: data_pagamento=113,835(5.2%)

📊 tb_geracao_distribuida — 3,510 linhas, 6 colunas
   id_uc:string, potencia_kwp:string, data_conexao:string, modalidade:string, energia_injetada_kwh_media_mensal:string, ingestion_datetime:timestamp
   Nulos: ✅ 0 em todas as colunas

📊 tb_inspecoes — 1,389 linhas

In [0]:
print(spark.table("`v-energy`.bronze.tb_atendimentos").schema["historico"].dataType)

spark.table("`v-energy`.bronze.tb_inspecoes").groupBy("sheet").count().show()

ArrayType(StructType([StructField('acao', StringType(), True), StructField('canal', StringType(), True), StructField('data', StringType(), True), StructField('sequencia', LongType(), True)]), True)
+-------+-----+
|    aba|count|
+-------+-----+
|out-nov|  495|
|    dez|  383|
|ago-set|  511|
+-------+-----+



In [0]:
# Teste 3: Validação focada na tabela tb_inspecoes (origem: inspecoes.xlsx)
# Verifica: contagem por aba, schema, nulos, duplicatas de id_inspecao e amostra

from pyspark.sql.functions import col, count, when, isnull, countDistinct
import pyspark.sql.functions as F

tabela = "`v-energy`.bronze.tb_inspecoes"
df = spark.table(tabela)

print("=" * 80)
print("TESTE 3: Validação da tabela tb_inspecoes (inspecoes.xlsx)")
print("=" * 80)

# 1. Contagem total e por aba
total = df.count()
print(f"\n1️⃣  Contagem total: {total:,} linhas (esperado: 1,389)")

contagem_abas = df.groupBy("sheet").count().orderBy("sheet").collect()
esperado_abas = {"ago-set": 511, "out-nov": 495, "dez": 383}
print(f"   Contagem por aba:")
ok_contagem = True
for row in contagem_abas:
    nome_aba = row["sheet"]
    qtd = row["count"]
    esp = esperado_abas.get(nome_aba)
    status = "✅" if qtd == esp else "⚠️"
    if qtd != esp:
        ok_contagem = False
    print(f"      {status} {nome_aba}: {qtd:,} (esperado: {esp:,})")
if ok_contagem:
    print("   ✅ Contagem por aba bate com o esperado")

# 2. Schema e tipos
print(f"\n2️⃣  Schema ({len(df.columns)} colunas):")
for c in df.columns:
    print(f"   {c}: {df.schema[c].dataType.simpleString()}")

colunas_esperadas = ["id_inspecao", "id_uc", "data", "inspetor", "resultado", 
                     "tipo_irregularidade", "kwh_recuperado", "sheet"]
faltando = set(colunas_esperadas) - set(df.columns)
extras = set(df.columns) - set(colunas_esperadas)
if not faltando and not extras:
    print("   ✅ Schema exatamente conforme esperado")
else:
    if faltando:
        print(f"   ⚠️ Colunas esperadas ausentes: {faltando}")
    if extras:
        print(f"   ⚠️ Colunas extras: {extras}")

# 3. Análise de nulos por coluna
print(f"\n3️⃣  Análise de nulos:")
null_counts = df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).collect()[0]
problemas_nulos = False
for c in df.columns:
    n = null_counts[c]
    pct = (n / total * 100) if total > 0 else 0
    if n > 0:
        problemas_nulos = True
        print(f"   ⚠️ {c}: {n:,} nulos ({pct:.1f}%)")
    else:
        print(f"   ✅ {c}: 0 nulos")
if not problemas_nulos:
    print("   ✅ Nenhuma coluna com nulos")

# 4. Duplicatas em id_inspecao
duplos = df.groupBy("id_inspecao").count().filter("count > 1").count()
print(f"\n4️⃣  Duplicatas de id_inspecao: {duplos}")
if duplos == 0:
    print("   ✅ Sem duplicatas")
else:
    print(f"   ⚠️ {duplos} id_inspecao repetido(s) — verificar")
    df.groupBy("id_inspecao").count().filter("count > 1").orderBy(col("count").desc()).show(10)

# 5. Valores distintos por coluna categórica
print(f"\n5️⃣  Valores distintos (colunas categóricas):")
for c in ["resultado", "tipo_irregularidade", "inspetor"]:
    n_distinct = df.select(countDistinct(c)).collect()[0][0]
    print(f"   {c}: {n_distinct} valor(es) distinto(s)")
    if n_distinct <= 15:
        valores = df.select(c).distinct().orderBy(c).collect()
        print(f"      → {', '.join(str(row[c]) for row in valores)}")

# 6. Amostra dos dados
print(f"\n6️⃣  Amostra (5 linhas aleatórias):")
display(df.orderBy(F.rand()).limit(5))

print(f"\n{'=' * 80}")

TESTE 3: Validação da tabela tb_inspecoes (inspecoes.xlsx)

1️⃣  Contagem total: 1,389 linhas (esperado: 1,389)
   Contagem por aba:
      ✅ ago-set: 511 (esperado: 511)
      ✅ dez: 383 (esperado: 383)
      ✅ out-nov: 495 (esperado: 495)
   ✅ Contagem por aba bate com o esperado

2️⃣  Schema (9 colunas):
   id_inspecao: string
   id_uc: string
   data: string
   inspetor: string
   resultado: string
   tipo_irregularidade: string
   kwh_recuperado: string
   aba: string
   ingestion_datetime: timestamp
   ⚠️ Colunas extras: {'ingestion_datetime'}

3️⃣  Análise de nulos:
   ✅ id_inspecao: 0 nulos
   ✅ id_uc: 0 nulos
   ✅ data: 0 nulos
   ✅ inspetor: 0 nulos
   ✅ resultado: 0 nulos
   ⚠️ tipo_irregularidade: 441 nulos (31.7%)
   ✅ kwh_recuperado: 0 nulos
   ✅ aba: 0 nulos
   ✅ ingestion_datetime: 0 nulos

4️⃣  Duplicatas de id_inspecao: 0
   ✅ Sem duplicatas

5️⃣  Valores distintos (colunas categóricas):
   resultado: 8 valor(es) distinto(s)
      → CONFIRMADA, IRREGULARIDADE CONFIRMAD

id_inspecao,id_uc,data,inspetor,resultado,tipo_irregularidade,kwh_recuperado,aba,ingestion_datetime
IN0000247,UC132971,2025-08-29,INSP-006,sem irregularidade,null,0.00,ago-set,2026-10-20T19:13:57.086Z
IN0000905,UC119308,2025-08-31,INSP-007,IRREGULARIDADE CONFIRMADA,LIGACAO CLANDESTINA,"519,80",ago-set,2026-10-20T19:13:57.086Z
IN0001137,UC119658,2025-11-15,INSP-033,IRREGULARIDADE CONFIRMADA,LIGACAO CLANDESTINA,575.00,out-nov,2026-10-20T19:13:57.086Z
IN0000176,UC144687,2025-10-13,INSP-039,SEM IRREGULARIDADE,null,0.00,out-nov,2026-10-20T19:13:57.086Z
IN0001208,UC102335,2025-09-26,INSP-038,IRREGULARIDADE CONFIRMADA,MEDIDOR ADULTERADO,981.20,ago-set,2026-10-20T19:13:57.086Z
